# 03 - Modelo de relación entre materias primas y equipos
Regresión lineal múltiple para identificar qué materias primas explican el precio de cada equipo y cuáles son ruido.

In [1]:
from pathlib import Path
import pandas as pd
import statsmodels.api as sm

DATA_PROCESSED = Path("../data/processed")

historico = pd.read_csv(
    DATA_PROCESSED / "historico_equipos_limpio.csv",
    parse_dates=["Date"]
)

In [2]:
materias_primas = ["Price_X", "Price_Y", "Price_Z"]

X_modelo = sm.add_constant(historico[materias_primas])
y_equipo1 = historico["Price_Equipo1"]

modelo_equipo1 = sm.OLS(y_equipo1, X_modelo).fit()
print(modelo_equipo1.summary())

                            OLS Regression Results                            
Dep. Variable:          Price_Equipo1   R-squared:                       0.995
Model:                            OLS   Adj. R-squared:                  0.995
Method:                 Least Squares   F-statistic:                 2.213e+05
Date:                Thu, 26 Nov 2026   Prob (F-statistic):               0.00
Time:                        09:16:20   Log-Likelihood:                -12463.
No. Observations:                3530   AIC:                         2.493e+04
Df Residuals:                    3526   BIC:                         2.496e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0831      0.796     -0.104      0.9

## Conclusión del Equipo 1

1. Las materias primas que explican el precio del Equipo 1 son X e Y, ya que ambas tienen un p-valor de 0,000 (menor a 0,05). El peso de X es aproximadamente 0,2 y el de Y es 0,8.
2. Al tener un peso de 0,8, Y es la materia prima más relevante para el Equipo 1.
3. La materia prima Z es ruido, debido a que su p-valor es de 0,999, casi el valor de 1, que sería su límite. Además, su coeficiente es casi cero (9,9 × 10⁻⁷), lo que muestra su poca relevancia.
4. El modelo es muy bueno, ya que su R² es de 0,995: explica el 99,5% del comportamiento del precio del Equipo 1 en los datos históricos.

In [5]:
materias_primas = ["Price_X", "Price_Y", "Price_Z"]

X_modelo = sm.add_constant(historico[materias_primas])
y_equipo2 = historico["Price_Equipo2"]

modelo_equipo2 = sm.OLS(y_equipo2, X_modelo).fit()
print(modelo_equipo2.summary())

                            OLS Regression Results                            
Dep. Variable:          Price_Equipo2   R-squared:                       0.992
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                 1.374e+05
Date:                Thu, 26 Nov 2026   Prob (F-statistic):               0.00
Time:                        14:16:12   Log-Likelihood:                -14720.
No. Observations:                3530   AIC:                         2.945e+04
Df Residuals:                    3526   BIC:                         2.947e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.2278      1.508      0.814      0.4

## Conclusión del Equipo 2

1. En el Equipo 2, las tres materias primas son relevantes: X, Y y Z tienen un p-valor de 0,000 (menor a 0,05) y un peso casi igual, de aproximadamente 0,33 cada una. Ninguna es ruido.
2. Anteriormente se veía que el Equipo 2 se comportaba de forma similar a Z, pero esto se debe a que Z es mucho más cara: aunque las tres pesan lo mismo por unidad, Z aporta la mayor parte del precio, y cada vez que Z se mueve, también se mueve el valor del equipo.
3. El modelo explica el 99,2% del comportamiento del precio del Equipo 2 en los datos históricos (R² = 0,992).
4. Que los tres pesos sean casi iguales sugiere que el precio del Equipo 2 se comporta como un promedio simple de las tres materias primas.

In [6]:
fecha_corte = "2022-01-01"

entrenamiento = historico[historico["Date"] < fecha_corte]
prueba = historico[historico["Date"] >= fecha_corte]

print("Entrenamiento:", entrenamiento["Date"].min().date(), "a", entrenamiento["Date"].max().date(), "-", len(entrenamiento), "días")
print("Prueba:", prueba["Date"].min().date(), "a", prueba["Date"].max().date(), "-", len(prueba), "días")

Entrenamiento: 2010-01-04 a 2021-12-31 - 3100 días
Prueba: 2022-01-03 a 2023-08-31 - 430 días


In [7]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

X_entrenamiento = sm.add_constant(entrenamiento[materias_primas])
X_prueba = sm.add_constant(prueba[materias_primas])

modelo_validacion_eq1 = sm.OLS(entrenamiento["Price_Equipo1"], X_entrenamiento).fit()
prediccion_eq1 = modelo_validacion_eq1.predict(X_prueba)

mae_eq1 = mean_absolute_error(prueba["Price_Equipo1"], prediccion_eq1)
mape_eq1 = mean_absolute_percentage_error(prueba["Price_Equipo1"], prediccion_eq1)

print(f"Equipo 1 - MAE: {mae_eq1:.2f} | MAPE: {mape_eq1:.2%}")

Equipo 1 - MAE: 8.04 | MAPE: 1.51%


In [9]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

X_entrenamiento = sm.add_constant(entrenamiento[materias_primas])
X_prueba = sm.add_constant(prueba[materias_primas])

modelo_validacion_eq2 = sm.OLS(entrenamiento["Price_Equipo2"], X_entrenamiento).fit()
prediccion_eq2 = modelo_validacion_eq2.predict(X_prueba)

mae_eq2 = mean_absolute_error(prueba["Price_Equipo2"], prediccion_eq2)
mape_eq2 = mean_absolute_percentage_error(prueba["Price_Equipo2"], prediccion_eq2)

print(f"Equipo 2 - MAE: {mae_eq2:.2f} | MAPE: {mape_eq2:.2%}")

Equipo 2 - MAE: 16.34 | MAPE: 1.50%


## Conclusión de la validación

1. El modelo se entrenó con los datos de 2010 a 2021 y se probó con los de 2022 a 2023, que nunca vio durante el aprendizaje. Los precios calculados fueron muy cercanos a los reales, lo que demuestra que el modelo sí aprendió la relación.
2. El error promedio (MAPE) fue de 1,51% para el Equipo 1 y de 1,50% para el Equipo 2.
3. Con esto se evidencia que el modelo es confiable para estimar el precio de los equipos con datos nuevos. Para proyectar el futuro, el siguiente paso es proyectar los precios de las materias primas.